# C2: Learning from labels (Strands head-only, LoRA, LoRA + head)

**Question.** How much does the pinned Strands Qwen3.5-2B v1 gain when it is trained on each dataset's labels, which part of the
model has to move (the pointer head, the LoRA adapter, or both), and how close does it come to methods that
use labels by design (TAR and the label-trained classifier)?

A 2 × 2 design on the joint question with probability features only: **zero-shot** (the B1 Strands joint arm,
no new run), **head only** (the released pointer head retrained on cached C1 `<answer>` and option-end vectors,
torso frozen), **LoRA only** (the released adapter trained, head frozen) and **LoRA + head**. Every trained
cell starts from Strands' released weights with its published recipe (`config/C/adaptation.yaml`, each value
with its source line). Targets are the true class; regression uses nine equal-frequency bins of each fit's
training rows mapped to Strands' nine ordered levels. Cross-fitting: three label-independent parts per outer
training fold; each part is scored by the fit trained on the other two; test rows average the three fits.

Common protocol (Group B): seven fixed predictors, the no-text baseline, 18 datasets, the same five outer
folds; raw gains within a task family (ΔAUC; ΔR²), datasets as the unit. TAR E5-small (two PCA recipes) and
the B2 label-trained classifier (joint, three encoders) are context; the C2 export carries them on identical
folds. The notebook reads only `output_JEVPFN/C/exp11_adaptation/results/group_c/c2_adaptation.zip`.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.notebook_display import display_json, display_local, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import group_c, group_c_views, learner_scores, study_export
from src.visualize import critical_difference, figures, group_c as gcfig, style
style.apply()
NOTEBOOK = "C/exp11_adaptation"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
display_json("Notebook execution identity", provenance())
frames, receipt = group_c.read("C2")
display_local(study_export.describe(receipt))
record({k: receipt.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, "Export identity")
WAITING = "Waiting for the C2 export (c2_adaptation.zip): nothing to show yet."
if frames:
    summary = group_c_views.overview(frames["dataset_gains"])
    contrasts = group_c_views.overview(frames["contrasts"])
    per_dataset = group_c_views.with_average(frames["dataset_gains"])
    names = group_c_views.dataset_names()

'Waiting for export: C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\C\\exp11_adaptation\\results\\group_c\\c2_adaptation.zip is absent. No measurements are shown.'

## 1. What is complete

Planned arms (3 trained cells per dataset and predictor) against complete five-fold arms, and the 810 planned
cross-fitted training fits (18 datasets × 5 folds × 3 parts × 3 cells) against the fits recorded in the export.
Per-fit diagnostics (epochs, validation loss, GPU hours, peak memory) are tabulated in `general/4_study_audit`.

**C0 — Strands qualification.** The three training configurations are first exercised on binary, multiclass and regression tasks. The head-only pilots use saved C1 vectors. The nine pilot receipts and measurements appear in the [study audit](../general/4_study_audit.ipynb). The counts above refer to production.

In [2]:
report_section("1. What is complete")
record(group_c.design_counts(), "Group C arms, configurations and C2 fits (plan, not results)")
if not frames:
    display_local(WAITING)
else:
    display_table("Planned versus complete five-fold arms per phase", group_c_views.coverage_summary(frames["coverage"]))
    display_table("Planned versus complete five-fold arms per predictor", frames["coverage"], preview_rows=14)
    display_table("Cross-fitted training fits per cell: planned, recorded, epochs, validation loss, GPU hours", group_c_views.fit_diagnostics(frames["fits"]))

'Waiting for the C2 export (c2_adaptation.zip): nothing to show yet.'

## 2. Summary: the 2 × 2 design

One figure: the interaction plot of the 2 × 2 (x = head released or trained; line = LoRA released or
trained), raw gain over no text per task family, with TAR and the label-trained classifier as horizontal
references. Points are equal-dataset means of the per-dataset mean over the seven predictors with
dataset-bootstrap 95% intervals.

Differences compare the complete training recipes. The adapted cells reset the released temperatures. Head-only training reuses cached canonical vectors, while the sequence-trained cells can augment the option order.

In [3]:
report_section("2. Summary: the 2 x 2 design")
if not frames:
    display_local(WAITING)
else:
    gcfig.render(save, gcfig.factorial_plot(summary, name="c2_factorial"))

'Waiting for the C2 export (c2_adaptation.zip): nothing to show yet.'

## 3. Detail: cell comparisons, label-using references and datasets

* **Cell comparisons**: the four approved fold-paired comparisons between the cells.
* **Context**: each cell minus TAR (two recipes) and minus the label-trained classifier (three encoders),
  in one grouped forest.
* **Per dataset**: the four cells and the two main references, dataset by dataset.
* **Supporting**: gain over no text of every cell, per predictor.

In [4]:
report_section("3. Detail: comparisons, references and datasets")
if not frames:
    display_local(WAITING)
else:
    gcfig.render(save, gcfig.c2_cells(contrasts, name="c2_cells"))
    gcfig.render(save, gcfig.c2_context(contrasts, name="c2_context"))
    gcfig.render(save, gcfig.c2_datasets(per_dataset, name="c2_datasets", names=names))
    gcfig.render(save, gcfig.c2_gains(summary, name="c2_gains"))
    display_table("Gain over no text per arm, predictor and task family (equal-dataset summary)", summary, preview_rows=20)
    display_table("Fold-paired comparisons (A minus B) per predictor and task family", contrasts, preview_rows=20)

'Waiting for the C2 export (c2_adaptation.zip): nothing to show yet.'

## 4. Statistics

For each predictor the approved exploratory analysis compares all six pairs of the four training recipes, with Holm adjustment across those six comparisons. The matched-gain table highlights the four prespecified recipe contrasts. Fold-level win rates, PAMA and ranks are descriptive.

In [5]:
report_section("4. Statistics")
framework = learner_scores.run_application("C2_strands_adaptation")
display_local(f"C2_strands_adaptation: {framework['status']} (framework.status = {framework['framework_status']})")
display_json("Framework application C2_strands_adaptation", learner_scores.summary_record(framework))
if framework["status"] == "compared":
    gcfig.render(save, critical_difference.cd_figures(framework))
else:
    display_local("Waiting for the export: no statistics figure is drawn.")

'C2_strands_adaptation: awaiting_export (framework.status = approved)'

'Waiting for the export: no statistics figure is drawn.'

## 5. Complete report

In [6]:
finish_report(NOTEBOOK, save.summary() + "\nOnly verified Group C exports were read; no inference, training or VSC change was performed.")

C/exp11_adaptation

Notebook execution identity
{
  "code_sha256": "8b4d9540fb90939d06c1177e768d032c9866801857bd06cfd36cb4fa3aea5040",
  "git_head": "e5d935fe73a631187c6d40435fbeb13c6743c494",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}

Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\C\exp11_adaptation\results\group_c\c2_adaptation.zip is absent. No measurements are shown.


Export identity
{
  "status": "awaiting_export",
  "path": "C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\C\\exp11_adaptation\\results\\group_